<a href="https://colab.research.google.com/github/vanessa0513/PraktikumKKAPraktikumAVD/blob/main/Analisis_Data_Kesehatan_dan_Pola_Tidur.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laporan & Dashboard Analisis Data Kesehatan dan Pola Tidur

**Sumber dataset:** Kaggle – `sleep_health_dataset.csv`  
**Tools:** Python, Pandas, NumPy, Matplotlib, Google Colab


## 1. Import Library dan Memuat Dataset

Jalankan sel berikut. Jika file belum tersedia di Google Colab, akan muncul tombol untuk mengunggah file CSV dari komputer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

# Upload file CSV dari komputer
uploaded = files.upload()

# Mengambil nama file CSV yang diunggah
file_name = next((name for name in uploaded.keys() if name.lower().endswith(".csv")), None)

if file_name is None:
    raise ValueError("File CSV belum diunggah. Silakan unggah sleep_health_dataset.csv.")

# Membaca dataset
df = pd.read_csv(file_name)

print("Dataset berhasil dimuat!")
print("Ukuran dataset:", df.shape)
display(df.head())

## 2. Eksplorasi dan Pembersihan Data

Memeriksa tipe data, nilai kosong, dan data duplikat. Baris duplikat dihapus jika ditemukan. Kolom `person_id` hanya berfungsi sebagai identitas responden, sehingga tidak digunakan sebagai variabel analisis.

In [ ]:
# Informasi umum dataset
print("Informasi dataset:")
df.info()

print("\\nJumlah data kosong setiap kolom:")
display(df.isnull().sum().to_frame("Jumlah Data Kosong"))

print("\\nJumlah baris duplikat:", df.duplicated().sum())

# Menghapus baris duplikat
df = df.drop_duplicates().copy()

# Memeriksa kembali data setelah pembersihan
print("\\nUkuran dataset setelah dibersihkan:", df.shape)
print("Total data kosong setelah pembersihan:", df.isnull().sum().sum())

# Ringkasan statistik awal
print("\\nRingkasan statistik awal:")
display(df.describe(include="all").T)

# Menampilkan jumlah setiap kategori risiko
print("\\nDistribusi risiko gangguan tidur:")
display(df["sleep_disorder_risk"].value_counts().to_frame("Jumlah Responden"))

## 3. Statistik Deskriptif

Statistik utama yang dihitung meliputi mean, median, standar deviasi, nilai minimum, maksimum, dan kuartil. Statistik kategori juga digunakan untuk mengetahui kelompok risiko yang paling banyak ditemukan.

In [ ]:
# Variabel numerik utama
kolom_numerik = [
    "age",
    "sleep_duration_hrs",
    "sleep_quality_score",
    "stress_score",
    "steps_that_day",
    "cognitive_performance_score"
]

# Tabel statistik deskriptif
statistik = df[kolom_numerik].describe().T
statistik = statistik[["count", "mean", "50%", "std", "min", "25%", "75%", "max"]]
statistik = statistik.rename(columns={
    "count": "Jumlah Data",
    "mean": "Rata-rata",
    "50%": "Median",
    "std": "Standar Deviasi",
    "min": "Minimum",
    "25%": "Q1",
    "75%": "Q3",
    "max": "Maksimum"
})

display(statistik.round(2))

# Statistik kategori
risiko_terbanyak = df["sleep_disorder_risk"].value_counts().idxmax()
jumlah_risiko_terbanyak = df["sleep_disorder_risk"].value_counts().max()
rata_tidur = df["sleep_duration_hrs"].mean()
rata_kualitas = df["sleep_quality_score"].mean()
rata_stres = df["stress_score"].mean()
rata_kognitif = df["cognitive_performance_score"].mean()

print("RINGKASAN STATISTIK UTAMA")
print(f"Jumlah responden             : {len(df):,}")
print(f"Rata-rata durasi tidur       : {rata_tidur:.2f} jam")
print(f"Rata-rata kualitas tidur     : {rata_kualitas:.2f}")
print(f"Rata-rata tingkat stres      : {rata_stres:.2f}")
print(f"Rata-rata performa kognitif  : {rata_kognitif:.2f}")
print(f"Risiko terbanyak             : {risiko_terbanyak} ({jumlah_risiko_terbanyak:,} responden)")

## 4. Visualisasi Data

Setiap grafik dilengkapi judul, label sumbu, serta insight singkat. Lima jenis grafik yang digunakan adalah Bar Chart, Line Chart, Scatter Plot, Histogram, dan Boxplot.

In [ ]:
# Grafik 1 - Bar Chart: rata-rata kualitas tidur berdasarkan risiko
urutan_risiko = ["Healthy", "Mild", "Moderate", "Severe"]
rata_kualitas_risiko = df.groupby("sleep_disorder_risk")["sleep_quality_score"].mean().reindex(urutan_risiko).dropna()

plt.figure(figsize=(8, 5))
plt.bar(rata_kualitas_risiko.index, rata_kualitas_risiko.values)
plt.title("Rata-rata Kualitas Tidur Berdasarkan Risiko Gangguan Tidur")
plt.xlabel("Kategori Risiko Gangguan Tidur")
plt.ylabel("Rata-rata Skor Kualitas Tidur")
plt.tight_layout()
plt.show()

print("INSIGHT 1:")
print("Grafik membandingkan kualitas tidur pada setiap kategori risiko.")
print("Perbedaan rata-rata dapat membantu melihat apakah kelompok dengan risiko lebih tinggi juga memiliki skor kualitas tidur yang lebih rendah.")

In [ ]:
# Grafik 2 - Line Chart: durasi tidur berdasarkan kelompok usia
batas_usia = [0, 20, 30, 40, 50, 60, 100]
label_usia = ["<=20", "21-30", "31-40", "41-50", "51-60", ">60"]
df["kelompok_usia"] = pd.cut(
    df["age"], bins=batas_usia, labels=label_usia, include_lowest=True
)
rata_tidur_usia = df.groupby("kelompok_usia", observed=False)["sleep_duration_hrs"].mean().dropna()

plt.figure(figsize=(8, 5))
plt.plot(rata_tidur_usia.index.astype(str), rata_tidur_usia.values, marker="o")
plt.title("Rata-rata Durasi Tidur Berdasarkan Kelompok Usia")
plt.xlabel("Kelompok Usia")
plt.ylabel("Rata-rata Durasi Tidur (Jam)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("INSIGHT 2:")
print("Grafik menunjukkan perubahan rata-rata durasi tidur pada kelompok usia yang berbeda.")
print("Kelompok usia dengan rata-rata durasi tidur paling rendah dapat menjadi perhatian dalam pembahasan pola tidur.")

In [ ]:
# Grafik 3 - Scatter Plot: durasi tidur dan performa kognitif
sampel = df.sample(n=min(3000, len(df)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(
    sampel["sleep_duration_hrs"],
    sampel["cognitive_performance_score"],
    alpha=0.35
)
plt.title("Hubungan Durasi Tidur dan Performa Kognitif")
plt.xlabel("Durasi Tidur (Jam)")
plt.ylabel("Skor Performa Kognitif")
plt.tight_layout()
plt.show()

korelasi = df["sleep_duration_hrs"].corr(df["cognitive_performance_score"])
print("INSIGHT 3:")
print(f"Nilai korelasi Pearson antara durasi tidur dan performa kognitif adalah {korelasi:.3f}.")
print("Nilai ini menunjukkan arah dan kekuatan hubungan linear, tetapi tidak membuktikan bahwa durasi tidur menyebabkan perubahan performa kognitif.")

In [ ]:
# Grafik 4 - Histogram: distribusi durasi tidur
plt.figure(figsize=(8, 5))
plt.hist(df["sleep_duration_hrs"], bins=25, edgecolor="black")
plt.title("Distribusi Durasi Tidur Responden")
plt.xlabel("Durasi Tidur (Jam)")
plt.ylabel("Frekuensi Responden")
plt.tight_layout()
plt.show()

print("INSIGHT 4:")
print("Histogram memperlihatkan sebaran durasi tidur dan rentang yang paling sering ditemukan.")
print("Sebaran yang lebar menunjukkan bahwa kebiasaan durasi tidur responden cukup beragam.")

In [ ]:
# Grafik 5 - Boxplot: durasi tidur berdasarkan risiko
data_boxplot = [
    df.loc[df["sleep_disorder_risk"] == kategori, "sleep_duration_hrs"].dropna()
    for kategori in urutan_risiko
    if kategori in df["sleep_disorder_risk"].unique()
]
label_boxplot = [
    kategori for kategori in urutan_risiko
    if kategori in df["sleep_disorder_risk"].unique()
]

plt.figure(figsize=(8, 5))
plt.boxplot(data_boxplot, labels=label_boxplot, showfliers=True)
plt.title("Sebaran Durasi Tidur Berdasarkan Risiko Gangguan Tidur")
plt.xlabel("Kategori Risiko Gangguan Tidur")
plt.ylabel("Durasi Tidur (Jam)")
plt.tight_layout()
plt.show()

print("INSIGHT 5:")
print("Boxplot memperlihatkan median, rentang kuartil, dan potensi nilai ekstrem durasi tidur pada setiap kelompok risiko.")
print("Perbedaan median dan sebaran antarkelompok dapat digunakan untuk membandingkan pola tidur responden.")

## 5. Deteksi Outlier Menggunakan Metode IQR

Outlier diperiksa pada variabel `sleep_duration_hrs`. Batas bawah dihitung dengan Q1 - 1,5 × IQR dan batas atas dengan Q3 + 1,5 × IQR.

In [ ]:
Q1 = df["sleep_duration_hrs"].quantile(0.25)
Q3 = df["sleep_duration_hrs"].quantile(0.75)
IQR = Q3 - Q1

batas_bawah = Q1 - (1.5 * IQR)
batas_atas = Q3 + (1.5 * IQR)

outlier = df[
    (df["sleep_duration_hrs"] < batas_bawah) |
    (df["sleep_duration_hrs"] > batas_atas)
]

print(f"Q1                 : {Q1:.2f} jam")
print(f"Q3                 : {Q3:.2f} jam")
print(f"IQR                : {IQR:.2f} jam")
print(f"Batas bawah        : {batas_bawah:.2f} jam")
print(f"Batas atas         : {batas_atas:.2f} jam")
print(f"Jumlah potensi outlier: {len(outlier):,}")
print(f"Persentase outlier : {len(outlier) / len(df) * 100:.2f}%")

display(outlier[["person_id", "age", "sleep_duration_hrs", "sleep_quality_score", "sleep_disorder_risk"]].head(10))

print("\\nINTERPRETASI:")
print("Nilai di luar batas IQR merupakan potensi outlier, bukan otomatis kesalahan data.")
print("Nilai ekstrem dapat terjadi karena variasi kebiasaan tidur, kondisi responden, atau kemungkinan kesalahan pencatatan. Data perlu diperiksa sebelum diputuskan untuk dihapus.")

## 6. Dashboard Ringkas

Dashboard menggabungkan empat grafik dalam satu figure (subplot), sesuai ketentuan proyek.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Bar chart kualitas tidur per kategori risiko
axes[0, 0].bar(rata_kualitas_risiko.index, rata_kualitas_risiko.values)
axes[0, 0].set_title("Kualitas Tidur Berdasarkan Risiko")
axes[0, 0].set_xlabel("Kategori Risiko")
axes[0, 0].set_ylabel("Rata-rata Skor")

# 2. Line chart durasi tidur per kelompok usia
axes[0, 1].plot(rata_tidur_usia.index.astype(str), rata_tidur_usia.values, marker="o")
axes[0, 1].set_title("Durasi Tidur Berdasarkan Usia")
axes[0, 1].set_xlabel("Kelompok Usia")
axes[0, 1].set_ylabel("Rata-rata Jam Tidur")
axes[0, 1].tick_params(axis="x", rotation=20)

# 3. Histogram durasi tidur
axes[1, 0].hist(df["sleep_duration_hrs"], bins=25, edgecolor="black")
axes[1, 0].set_title("Distribusi Durasi Tidur")
axes[1, 0].set_xlabel("Durasi Tidur (Jam)")
axes[1, 0].set_ylabel("Frekuensi")

# 4. Scatter plot tidur dan performa kognitif
axes[1, 1].scatter(sampel["sleep_duration_hrs"], sampel["cognitive_performance_score"], alpha=0.35)
axes[1, 1].set_title("Tidur vs Performa Kognitif")
axes[1, 1].set_xlabel("Durasi Tidur (Jam)")
axes[1, 1].set_ylabel("Skor Performa Kognitif")

fig.suptitle("DASHBOARD ANALISIS KESEHATAN DAN POLA TIDUR", fontsize=16)
plt.tight_layout()
plt.show()

## 7. Temuan Utama dan Laporan Rekomendasi (200–300 Kata)

Bagian ini membuat rekomendasi awal secara otomatis berdasarkan hasil perhitungan dataset. Periksa kembali hasilnya setelah seluruh sel dijalankan.

In [ ]:
# Membandingkan rata-rata berdasarkan kategori risiko
ringkasan_risiko = df.groupby("sleep_disorder_risk")[
    ["sleep_duration_hrs", "sleep_quality_score", "stress_score", "cognitive_performance_score"]
].mean().reindex(urutan_risiko).dropna()

print("RATA-RATA BERDASARKAN KATEGORI RISIKO:")
display(ringkasan_risiko.round(2))

# Membuat rekomendasi berdasarkan hasil data
jumlah_healthy = int((df["sleep_disorder_risk"] == "Healthy").sum())
persen_healthy = jumlah_healthy / len(df) * 100
persen_outlier = len(outlier) / len(df) * 100

laporan = f"""
Berdasarkan analisis terhadap {len(df):,} responden, rata-rata durasi tidur tercatat sebesar {rata_tidur:.2f} jam, sedangkan median durasi tidur sebesar {df['sleep_duration_hrs'].median():.2f} jam. Rata-rata skor kualitas tidur adalah {rata_kualitas:.2f}, tingkat stres {rata_stres:.2f}, dan performa kognitif {rata_kognitif:.2f}. Hasil ini memberikan gambaran umum mengenai kebiasaan tidur dan kondisi responden dalam dataset.

Kategori risiko yang paling banyak ditemukan adalah {risiko_terbanyak}, dengan jumlah {jumlah_risiko_terbanyak:,} responden. Perbandingan kelompok menunjukkan bahwa rata-rata durasi tidur, kualitas tidur, tingkat stres, dan performa kognitif dapat berbeda menurut kategori risiko. Temuan tersebut perlu diperhatikan sebagai pola hubungan dalam data, bukan sebagai bukti bahwa satu faktor secara langsung menyebabkan faktor lainnya.

Analisis korelasi antara durasi tidur dan performa kognitif menghasilkan nilai {korelasi:.3f}. Nilai tersebut dapat digunakan untuk menjelaskan arah hubungan linear kedua variabel. Sementara itu, metode IQR menemukan {len(outlier):,} potensi outlier durasi tidur atau sekitar {persen_outlier:.2f}% dari seluruh responden. Nilai ekstrem tersebut sebaiknya diperiksa lebih lanjut dan tidak langsung dihapus karena mungkin mencerminkan variasi kondisi responden.

Berdasarkan temuan tersebut, rekomendasi yang dapat diberikan adalah meningkatkan edukasi mengenai kebiasaan tidur yang teratur, memperhatikan kelompok responden dengan kategori risiko lebih tinggi, serta mendorong pengelolaan stres dan keseimbangan aktivitas harian. Analisis lanjutan dapat membandingkan pola tidur berdasarkan usia, pekerjaan, aktivitas fisik, dan penggunaan kafein sebelum tidur. Pihak yang menggunakan hasil analisis perlu mengingat bahwa dataset ini bersifat observasional sehingga hasilnya tidak dapat digunakan untuk menyimpulkan hubungan sebab-akibat atau menggantikan pemeriksaan tenaga kesehatan.
"""

jumlah_kata = len(laporan.split())
print(laporan)
print("\\nJumlah kata laporan:", jumlah_kata)
if not 200 <= jumlah_kata <= 300:
    print("PERHATIAN: Sesuaikan laporan agar berada pada rentang 200–300 kata.")

## 8. Kesimpulan Proyek

Berdasarkan hasil analisis terhadap 100.000 data responden dalam dataset Sleep Health, dapat disimpulkan bahwa pola tidur merupakan salah satu aspek penting yang perlu diperhatikan dalam menjaga kesehatan dan aktivitas sehari-hari. Hasil analisis menunjukkan rata-rata durasi tidur responden sebesar 6,42 jam, dengan rata-rata kualitas tidur sebesar 4,87 dan tingkat stres sebesar 5,73.

Dari distribusi risiko gangguan tidur, sebanyak 54.156 responden termasuk kategori Healthy, 33.479 kategori Mild, 8.299 kategori Moderate, dan 4.066 kategori Severe. Hasil visualisasi juga membantu memperlihatkan perbedaan pola tidur berdasarkan kelompok usia dan kategori risiko, serta hubungan antara durasi tidur dengan performa kognitif.

Melalui metode IQR, analisis dapat mengidentifikasi potensi outlier pada durasi tidur yang perlu diperiksa lebih lanjut. Nilai ekstrem tersebut tidak selalu menunjukkan kesalahan data, tetapi dapat menggambarkan variasi kebiasaan tidur responden.

Secara keseluruhan, proyek ini menunjukkan bahwa analisis data dapat membantu memahami pola tidur dan faktor-faktor yang berkaitan dengannya. Oleh karena itu, masyarakat disarankan untuk menjaga jadwal tidur yang teratur, mengelola stres, serta menyeimbangkan aktivitas harian dengan waktu istirahat yang cukup. Hasil analisis ini diharapkan dapat meningkatkan kesadaran mengenai pentingnya kualitas tidur bagi kesehatan dan performa kognitif.




## 9. Menyimpan Dataset Hasil Pembersihan (Opsional)

Sel berikut menyimpan data yang sudah dibersihkan ke file CSV.

In [ ]:
df.to_csv("sleep_health_dataset_cleaned.csv", index=False)
print("File hasil pembersihan berhasil disimpan.")
files.download("sleep_health_dataset_cleaned.csv")